# CS-781 Phase 2 (L-NER) — `p2_16b_ensemble_cross_arch`: span-voting ensemble of the models already trained

**Variant B: cross-architecture** (MoE, GQA and LSTM-family models, including the two distilled students p2_15a/p2_15b). The grid compares `one_per_model` (3 members) with `two_per_model` (6 members) and the top-N sets, plus a `no_lstm_family` group, so one run answers whether a second model per architecture helps and whether the distilled LSTM student adds anything to MoE + GQA.

No new training and **no GPU needed** (set the accelerator to *None*). The notebook takes the entity spans predicted by several of your models and keeps a span when enough of them agree:
* **fold2 check** — the fold2 predictions of the fold1-trained models are in your bucket; the notebook scores every single model and every ensemble setting (which models, vote threshold, exact-span voting vs boundary voting, weights, abbreviation-period normalisation) and reports an **honest estimate** (setting chosen on half of the fold2 documents, scored on the other half);
* **test predictions** — built from the `prediction.json` inside the submission zips of the fold1+fold2 refits, so you must attach them (see below).

**Before running — attach your zips:** on Kaggle, *Add Data → Upload → New dataset*, upload the submission `.zip` files you already made (MoE p2_06/09/12/13/14, GQA, LSTM, MoR, GDN ones; the more the better, a member without a zip is only used for the fold2 analysis) and add that dataset to this notebook. File names may be the original ones (`submission_p2s_gqa160_adapter_bilstm.zip`) or the Codabench ones (`p2s_gqa160_adapter_bilstm.zip`).

**Kaggle:** secrets `HF`, `HF_WRITE`; Accelerator *None* is enough. Takes a few minutes. Writes up to two zips (`..._A` = best setting, `..._B` = best setting with a different member set).

## 1. Settings

In [ ]:
import json
# =====================================================================================================
# ENSEMBLE SETTINGS -- every value is YOUR choice
# =====================================================================================================
RUN_NAME = "p2_16b_ensemble_cross_arch"
QUICK_TEST = False
# (key, fold2 predictions of the fold1-trained model in the bucket, name of the submission zip of the fold1+fold2 refit WITHOUT ".zip"
#  and without a leading "submission_")  -- a member without its zip is used for the fold2 analysis only.
MEMBERS = [
    ("moe13_adapter_bilstm", "p2_13_moe_hc_adapter_bilstm_clsw_heldout_fold2_pred.json", "p2_13_moe_hc_adapter_bilstm_clsw"),
    ("moe14_lorma_bilstm", "p2_14_moe_hc_lorma_bilstm_clsw_heldout_fold2_pred.json", "p2_14_moe_hc_lorma_bilstm_clsw"),
    ("gqa_adapter_bilstm", "p2s_gqa160__adapter_bilstm__fold1tofold2_pred.json", "p2s_gqa160_adapter_bilstm"),
    ("gqa_pfeiffer_bilstm", "p2s_gqa160__adapter_pfeiffer_bilstm__fold1tofold2_pred.json", "p2s_gqa160_adapter_pfeiffer_bilstm"),
    ("lstm_partial_top1", "p2s_lstm84__partial_ft_top1_bilstm__fold1tofold2_pred.json", "p2s_lstm84_partial_ft_top1_bilstm"),
    ("lstm_adapter_bilstm", "p2s_lstm84__adapter_bilstm__fold1tofold2_pred.json", "p2s_lstm84_adapter_bilstm"),
    ("kd15b_unlabelled", "p2_15b_kd_moe13_to_lstm_plus_unlabelled_heldout_fold2_pred.json", "p2_15b_kd_moe13_to_lstm_plus_unlabelled"),
    ("kd15a_labelled", "p2_15a_kd_moe13_to_lstm_labelled_only_heldout_fold2_pred.json", "p2_15a_kd_moe13_to_lstm_labelled_only"),
]
# member groups tried besides "the best N by fold2 F1" (keys from MEMBERS)
GROUPS = {
    "one_per_model": ["moe13_adapter_bilstm", "gqa_adapter_bilstm", "kd15b_unlabelled"],
    "two_per_model": ["moe13_adapter_bilstm", "moe14_lorma_bilstm", "gqa_adapter_bilstm", "gqa_pfeiffer_bilstm", "kd15b_unlabelled", "lstm_partial_top1"],
    "no_lstm_family": ["moe13_adapter_bilstm", "moe14_lorma_bilstm", "gqa_adapter_bilstm", "gqa_pfeiffer_bilstm"],
}
TOP_N_LIST = [3, 4, 5, 99]                      # "the best N members by fold2 F1" (99 = all)
TAUS = [round(0.15 + 0.05 * i, 2) for i in range(14)]   # a span is kept when its (weighted) vote share >= tau
MODES = ["exact", "boundary"]                        # exact = vote on (start, end, label); boundary = vote on (start, end), label = most voted
WEIGHTS = ["equal", "f1"]                            # every member counts 1, or in proportion to its fold2 F1
DOT_RULES = [False, True]                            # normalise abbreviation periods ("Ltd" vs "Ltd.") in every member before voting
INPUT_MODE = "none"


## 2. Setup: login, bucket

In [ ]:
!pip install -q -U "huggingface_hub>=1.5"
import os, json, shutil, subprocess, sys, time, zipfile, random, math, gc
from collections import Counter, defaultdict
import torch, torch.nn as nn, torch.nn.functional as F
from huggingface_hub import hf_hub_download, login, download_bucket_files, batch_bucket_files
from kaggle_secrets import UserSecretsClient

NB_START = time.time()
WORK = "/kaggle/working"
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_BUCKET = "kavincg23/cs781-models"
HF_CHECKPOINT_REPO = "kavincg23/cs781-checkpoints"   # old repo, read-only fallback
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
HF_WRITE_TOKEN = user_secrets.get_secret("HF_WRITE")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")


def get_dataset_file(filename):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename, repo_type="dataset", token=HF_TOKEN)


def fetch_model_file(name, local_path=None):
    local_path = local_path or f"{WORK}/{name}"
    if os.path.exists(local_path):
        return local_path
    part = local_path + ".part"
    try:
        download_bucket_files(HF_BUCKET, files=[(name, part)], raise_on_missing_files=True, token=HF_WRITE_TOKEN)
        os.replace(part, local_path)
        return local_path
    except Exception:
        if os.path.exists(part):
            os.remove(part)
    # fallback: the OLD model repo (the Phase-1 notebooks do the same; the tokenizer only lives there)
    p = hf_hub_download(repo_id=HF_CHECKPOINT_REPO, filename=name, repo_type="model", token=HF_WRITE_TOKEN)
    shutil.copy(p, local_path)
    print(f"(found {name} in the old repo {HF_CHECKPOINT_REPO})")
    return local_path


def upload_model_file(local_path, name=None):
    name = name or os.path.basename(local_path)
    try:
        batch_bucket_files(HF_BUCKET, add=[(local_path, name)], token=HF_WRITE_TOKEN)
        print(f"uploaded {name} to bucket {HF_BUCKET}")
    except Exception as e:
        print(f"(upload of {name} failed: {type(e).__name__}: {e}; the file is still in {WORK})")


## 3. Phase-2 data

In [ ]:
P2 = "Phase_2_LNER"
def load_json(p):
    with open(p, encoding="utf-8") as f:
        return json.load(f)

LABELS = [l.strip() for l in open(get_dataset_file(f"{P2}/Train/labels.txt"), encoding="utf-8") if l.strip()]
TEXTS = {f: load_json(get_dataset_file(f"{P2}/Train/{f}-text.json")) for f in ("fold1", "fold2")}
GOLDS = {f: load_json(get_dataset_file(f"{P2}/Train/{f}-gold.json")) for f in ("fold1", "fold2")}
TEST_TEXTS = load_json(get_dataset_file(f"{P2}/Test/fold3-text.json"))
for f in TEXTS:
    assert set(TEXTS[f]) == set(GOLDS[f])
    assert all(TEXTS[f][d][e[0]:e[1]] == e[3] for d in GOLDS[f] for e in GOLDS[f][d]), "gold offsets do not match the text"
if QUICK_TEST:
    for f in TEXTS:
        keep = list(TEXTS[f])[:2]
        TEXTS[f] = {k: TEXTS[f][k] for k in keep}; GOLDS[f] = {k: GOLDS[f][k] for k in keep}
    TEST_TEXTS = {k: TEST_TEXTS[k] for k in list(TEST_TEXTS)[:3]} | {k: v for k, v in TEST_TEXTS.items() if not v.strip()}
print("labels:", LABELS)
print({f: (len(TEXTS[f]), sum(len(v) for v in GOLDS[f].values())) for f in TEXTS}, "| test docs:", len(TEST_TEXTS))


## 4. Strict scorer and error breakdown (same code as in the other notebooks)

In [ ]:
# ===== Phase 2 L-NER: character offsets for the Phase-1 BPE tokenizer, BIO tags, span decoding =====
# Ancillary pipeline code (data handling / alignment / decoding). It does NOT change the tokenizer: the token ids
# produced here are exactly BPETokenizer.encode(text) (checked on every document by check_encode_identical).

LABELS_DEFAULT = ["APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"]
_END_MARK = "</w>"


def iter_words(text):
    """(start_char, end_char) of every whitespace-separated word, using exactly str.split()'s notion of whitespace
    (str.isspace), so the words are the same ones BPETokenizer.encode() sees."""
    n, i = len(text), 0
    while i < n:
        while i < n and text[i].isspace():
            i += 1
        if i >= n:
            break
        j = i
        while j < n and not text[j].isspace():
            j += 1
        yield i, j
        i = j


import re as _re
PIECE_MODES = {
    "none": None,                                                   # exactly the Phase-1 tokenization
    "punct": _re.compile(r"\w+|[^\w\s]"),                          # split every punctuation char off
    "punct_digit": _re.compile(r"[^\W\d_]+|\d+|_+|[^\w\s]"),       # also split letters from digits
}


def iter_pieces(text, mode="none"):
    """Words as the tokenizer sees them. mode 'none' = whitespace words (Phase-1 behaviour); the other modes split
    each whitespace word further, which changes the token ids versus pretraining (a design choice to measure)."""
    rx = PIECE_MODES[mode]
    for ws, we in iter_words(text):
        if rx is None:
            yield ws, we
        else:
            for m in rx.finditer(text, ws, we):
                yield m.start(), m.end()


def encode_with_offsets(tok, text, mode="none"):
    """Return (ids, spans): ids == tok.encode(text); spans[k] = (start_char, end_char) of token k in `text`
    (end exclusive). A token's symbols are UTF-8 bytes held as chr(b); when a token boundary falls inside a
    multi-byte character, that character is given to the token holding its first byte (the next token then starts
    after it; if a token holds only continuation bytes it gets an empty span at that position)."""
    unk = tok.stoi["<unk>"]
    ids, spans = [], []
    for ws, we in iter_pieces(text, mode):
        word = text[ws:we]
        syms = tok._tokenize_word(word)
        cached = tok._id_cache.get(word)
        if cached is None:
            cached = [tok.stoi.get(t, unk) for t in syms]
            tok._id_cache[word] = cached
        # byte offset -> char offset inside the word
        byte_start_of_char = []
        b = 0
        for ch in word:
            byte_start_of_char.append(b)
            b += len(ch.encode("utf-8"))
        total_bytes = b
        byte_to_char = [0] * (total_bytes + 1)
        for ci, bs in enumerate(byte_start_of_char):
            nb = (byte_start_of_char[ci + 1] if ci + 1 < len(byte_start_of_char) else total_bytes) - bs
            for k in range(nb):
                byte_to_char[bs + k] = ci if k == 0 else ci + 1      # a continuation byte "belongs" after its char start
        byte_to_char[total_bytes] = len(word)
        bpos = 0
        for sym in syms:
            core = sym[:-len(_END_MARK)] if sym.endswith(_END_MARK) else sym
            nbytes = len(core)                                       # each symbol char is one byte
            cs = byte_to_char[bpos] if bpos < total_bytes else len(word)
            bpos += nbytes
            ce = byte_to_char[bpos] if bpos < total_bytes else len(word)
            if ce < cs:
                ce = cs
            spans.append((ws + cs, ws + ce))
        ids.extend(cached)
    return ids, spans


def check_encode_identical(tok, docs):
    """Assert our ids equal tok.encode on every doc and that spans are monotone and inside the text."""
    for did, text in docs.items():
        ids, spans = encode_with_offsets(tok, text)
        ref = tok.encode(text)
        assert ids == ref, f"doc {did}: ids differ from BPETokenizer.encode"
        prev = 0
        for (s, e) in spans:
            assert prev <= s <= e <= len(text), f"doc {did}: bad span {(s, e)}"
            prev = e
    return True


# ---------------- BIO tags ----------------
IGNORE = -100        # label for tokens that cover no character (loss ignores them; decoding skips them)
def bio_tag_names(labels):
    return ["O"] + [f"{p}-{l}" for l in labels for p in ("B", "I")]


def gold_to_token_tags(spans, ents, labels):
    """Token tag ids (0 = O, 1+2*li = B-label, 2+2*li = I-label). A token is inside an entity if its char span
    overlaps the entity. Also returns how many entities have a start/end that is NOT on a token boundary."""
    tag = [0 if e > s else IGNORE for s, e in spans]                # empty-span tokens (e.g. a lone '</w>') -> ignored
    lab_idx = {l: i for i, l in enumerate(labels)}
    starts = {s for s, e in spans if e > s}
    ends = {e for s, e in spans if e > s}
    misaligned = []
    ents_sorted = sorted(ents, key=lambda x: (x[0], x[1]))
    k = 0
    for ent in ents_sorted:
        s, e, lab = ent[0], ent[1], ent[2]
        li = lab_idx[lab]
        if s not in starts or e not in ends:
            misaligned.append(ent)
        while k < len(spans) and spans[k][1] <= s:
            k += 1
        j, first = k, True
        while j < len(spans) and spans[j][0] < e:
            if spans[j][1] > spans[j][0]:                            # skip empty (continuation-byte) tokens
                if tag[j] == 0:                                     # never overwrite another entity's token
                    tag[j] = 1 + 2 * li if first else 2 + 2 * li
                first = False
            j += 1
    return tag, misaligned


def token_tags_to_spans(spans, tag, labels):
    """BIO tag ids -> [(start_char, end_char, label)] with lenient repair: an I- tag that does not continue an
    entity of the same label starts a new one."""
    out, cur = [], None
    for (s, e), t in zip(spans, tag):
        if e <= s or t == IGNORE:
            continue
        if t == 0:
            if cur:
                out.append(tuple(cur)); cur = None
            continue
        li, is_b = (t - 1) // 2, (t - 1) % 2 == 0
        lab = labels[li]
        if cur and not is_b and cur[2] == lab:
            cur[1] = e
        else:
            if cur:
                out.append(tuple(cur))
            cur = [s, e, lab]
    if cur:
        out.append(tuple(cur))
    return out


# ---------------- boundary repair (character-level, after token decoding) ----------------
TRAIL_PUNCT = set(",.;:)]}'\"’”`-/")
LEAD_PUNCT = set("([{'\"‘“`-:./")


def learn_trim_table(docs, golds, tok_spans_by_doc, labels):
    """Data-driven boundary repair learned on TRAINING docs only. For each (label, trailing char) seen at the end of
    an oracle token span, count how often the gold entity excludes that char -> trim if excluded > kept.
    Same for leading chars. Returns (trail_rule, lead_rule) dicts {(label, ch): True/False}."""
    from collections import Counter
    t_ex, t_keep, l_ex, l_keep = Counter(), Counter(), Counter(), Counter()
    for did, ents in golds.items():
        text, spans = docs[did], tok_spans_by_doc[did]
        tag, _ = gold_to_token_tags(spans, ents, labels)
        pred = token_tags_to_spans(spans, tag, labels)
        # match each oracle span to the gold entity it covers (same label, contained)
        gl = sorted((e[0], e[1], e[2]) for e in ents)
        for (s, e, lab) in pred:
            cands = [g for g in gl if g[2] == lab and g[0] >= s and g[1] <= e]
            if len(cands) != 1:
                continue
            gs, ge, _ = cands[0]
            # trailing chars of the token span beyond gold end
            for p in range(e - 1, s - 1, -1):
                ch = text[p]
                if ch not in TRAIL_PUNCT:
                    break
                if p >= ge:
                    t_ex[(lab, ch)] += 1
                else:
                    t_keep[(lab, ch)] += 1
                    break
            for p in range(s, e):
                ch = text[p]
                if ch not in LEAD_PUNCT:
                    break
                if p < gs:
                    l_ex[(lab, ch)] += 1
                else:
                    l_keep[(lab, ch)] += 1
                    break
    trail = {k: t_ex[k] > t_keep[k] for k in set(t_ex) | set(t_keep)}
    lead = {k: l_ex[k] > l_keep[k] for k in set(l_ex) | set(l_keep)}
    return trail, lead


def repair_span(text, s, e, lab, trail_rule, lead_rule):
    while e > s + 1 and text[e - 1] in TRAIL_PUNCT and trail_rule.get((lab, text[e - 1]), False):
        e -= 1
    while s < e - 1 and text[s] in LEAD_PUNCT and lead_rule.get((lab, text[s]), False):
        s += 1
    return s, e


def finalize_doc_entities(text, raw_spans, trail_rule=None, lead_rule=None):
    """Repair, drop empty/invalid, de-duplicate (the evaluator FAILS on duplicate (start,end,label)), sort."""
    seen, out = set(), []
    for s, e, lab in raw_spans:
        if trail_rule is not None:
            s, e = repair_span(text, s, e, lab, trail_rule, lead_rule or {})
        s, e = int(s), int(e)
        if not (0 <= s < e <= len(text)):
            continue
        key = (s, e, lab)
        if key in seen:
            continue
        seen.add(key)
        out.append([s, e, lab])
    out.sort()
    return out


# ---------------- strict scorer (same arithmetic as evaluate_phase2.py) + error breakdown ----------------
def strict_scores(golds, preds, labels):
    stats = {l: [0, 0, 0] for l in labels}
    for did, g in golds.items():
        gs = {(x[0], x[1], x[2]) for x in g}
        ps = {(x[0], x[1], x[2]) for x in preds.get(did, [])}
        for l in labels:
            a = {x for x in gs if x[2] == l}; b = {x for x in ps if x[2] == l}
            stats[l][0] += len(a & b); stats[l][1] += len(b - a); stats[l][2] += len(a - b)
    per = {}
    for l, (tp, fp, fn) in stats.items():
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f = 2 * p * r / (p + r) if p + r else 0.0
        per[l] = dict(P=p, R=r, F1=f, tp=tp, fp=fp, fn=fn)
    n = len(labels)
    return dict(macro_F1=sum(v["F1"] for v in per.values()) / n,
                macro_P=sum(v["P"] for v in per.values()) / n,
                macro_R=sum(v["R"] for v in per.values()) / n, per_label=per)


def error_breakdown(golds, preds):
    """Classify each wrong prediction: boundary error (overlaps a gold entity of the same label), label error (same
    exact span, other label), boundary+label, or spurious (overlaps no gold entity). Also gold entities missed."""
    from collections import Counter
    c = Counter()
    for did, g in golds.items():
        gs = {(x[0], x[1], x[2]) for x in g}
        for (s, e, l) in {(x[0], x[1], x[2]) for x in preds.get(did, [])}:
            if (s, e, l) in gs:
                c["correct"] += 1; continue
            same_span = [x for x in gs if x[0] == s and x[1] == e]
            over = [x for x in gs if x[0] < e and s < x[1]]
            if same_span:
                c["label_error"] += 1
            elif any(x[2] == l for x in over):
                c["boundary_error"] += 1
            elif over:
                c["boundary_and_label_error"] += 1
            else:
                c["spurious"] += 1
        ps = {(x[0], x[1]) for x in preds.get(did, [])}
        for x in gs:
            if not any(p[0] < x[1] and x[0] < p[1] for p in ps):
                c["missed_entirely"] += 1
    return dict(c)


# ---------------- sliding windows ----------------
def make_windows(n_tokens, window, stride):
    """Start indices of windows of length `window` every `stride` tokens, the last one flush with the end."""
    if n_tokens <= window:
        return [0]
    starts = list(range(0, n_tokens - window + 1, stride))
    if starts[-1] + window < n_tokens:
        starts.append(n_tokens - window)
    return starts


## 5. Voting, abbreviation-period normalisation

In [ ]:
import re
from bisect import bisect_left
from collections import defaultdict, Counter


def learn_dot_rule(texts, golds):
    """For every gold entity count whether the entity INCLUDES a final period or stops just before one, per
    (label, last word) and per (label, word shape). Learned on training documents only."""
    inc, exc = Counter(), Counter()
    for did, ents in golds.items():
        t = texts[did]
        for e in ents:
            s, en, lab = e[0], e[1], e[2]
            body = t[s:en]
            if body.endswith("."):
                m = re.search(r"([A-Za-z]+)\.$", body); kind = inc
                word = m.group(1) if m else ""
            elif t[en:en + 1] == ".":
                m = re.search(r"([A-Za-z]+)$", body); kind = exc
                word = m.group(1) if m else ""
            else:
                continue
            if not word:
                continue
            kind[(lab, word.lower())] += 1
            kind[(lab, "shape:" + ("short" if len(word) <= 3 else "long") + ("cap" if word[0].isupper() else "low"))] += 1
    return inc, exc


def apply_dot_rule(texts, preds, rule):
    inc, exc = rule
    out = {}
    for did, ents in preds.items():
        t, new = texts[did], set()
        for (s, e, lab) in ents:
            if not t[e - 1:e] == "." and t[e:e + 1] == ".":
                m = re.search(r"([A-Za-z]+)$", t[s:e])
                if m:
                    w = m.group(1)
                    k1, k2 = (lab, w.lower()), (lab, "shape:" + ("short" if len(w) <= 3 else "long") + ("cap" if w[0].isupper() else "low"))
                    if inc[k1] + exc[k1] >= 2:
                        go = inc[k1] > exc[k1]
                    elif inc[k2] + exc[k2] >= 3:
                        go = inc[k2] > exc[k2]
                    else:
                        go = False
                    if go:
                        e += 1
            new.add((s, e, lab))
        out[did] = sorted(new)
    return out


def vote(pred_list, weights, tau, mode, doc_ids):
    W = float(sum(weights))
    out = {}
    for d in doc_ids:
        cnt = defaultdict(float)
        for p, w in zip(pred_list, weights):
            for (s, e, l) in set(p.get(d, [])):
                cnt[(s, e, l)] += w
        if mode == "exact":
            cands = [(v, k) for k, v in cnt.items() if v >= tau * W - 1e-9]
        else:
            by = defaultdict(dict)
            for (s, e, l), v in cnt.items():
                by[(s, e)][l] = v
            cands = []
            for (s, e), dl in by.items():
                sup = sum(dl.values())
                lab = max(dl, key=lambda x: (dl[x], x))
                if sup >= tau * W - 1e-9:
                    cands.append((dl[lab] + 1e-3 * sup, (s, e, lab)))
        cands.sort(key=lambda t: (-t[0], -(t[1][1] - t[1][0]), t[1][0], t[1][2]))
        starts, ends, res = [], [], []                 # accepted intervals, non-overlapping (the gold has no overlaps)
        for v, (s, e, l) in cands:
            if any(not (e <= ts or s >= te) for ts, te in zip(starts, ends)):
                continue
            starts.append(s); ends.append(e); res.append((s, e, l))
        out[d] = sorted(res)
    return out


## 6. Load the members' fold2 predictions

In [ ]:
# ---- fold2 predictions of every member (fold1-trained models) --------------------------------------------------------
GOLD2 = {d: [tuple(e[:3]) for e in GOLDS["fold2"][d]] for d in GOLDS["fold2"]}
FOLD2 = {}
for key, fname, zname in MEMBERS:
    try:
        p = json.load(open(fetch_model_file(fname)))
        assert set(p) >= set(GOLD2), "missing documents"
        FOLD2[key] = {d: [tuple(x[:3]) for x in p[d]] for d in GOLD2}
    except Exception as ex:
        print(f"  skipped {key}: {type(ex).__name__}: {str(ex)[:100]}")
SCORES = {k: strict_scores(GOLD2, p, LABELS) for k, p in FOLD2.items()}
print(f"{len(FOLD2)} members loaded; single-model strict macro-F1 on fold2:")
for k in sorted(SCORES, key=lambda k: -SCORES[k]["macro_F1"]):
    s = SCORES[k]
    print(f"  {k:24s} F1 {s['macro_F1']:.4f}  P {s['macro_P']:.4f}  R {s['macro_R']:.4f}  predicted spans {sum(len(v) for v in FOLD2[k].values())}")
BEST_SINGLE = max(SCORES, key=lambda k: SCORES[k]["macro_F1"])
print("best single model:", BEST_SINGLE, round(SCORES[BEST_SINGLE]["macro_F1"], 4))


## 7. Search the ensemble settings on fold2

In [ ]:
# ---- search the ensemble settings on fold2 ----------------------------------------------------------------------------
DOT1 = learn_dot_rule(TEXTS["fold1"], GOLDS["fold1"])                    # learned on fold1 only (fold2 is the evaluation fold)
FOLD2_NORM = {k: apply_dot_rule(TEXTS["fold2"], p, DOT1) for k, p in FOLD2.items()}
print("abbreviation-period normalisation on single models (fold2 F1 before -> after):")
for k in sorted(SCORES, key=lambda k: -SCORES[k]["macro_F1"])[:6]:
    print(f"  {k:24s} {SCORES[k]['macro_F1']:.4f} -> {strict_scores(GOLD2, FOLD2_NORM[k], LABELS)['macro_F1']:.4f}")

order = sorted(FOLD2, key=lambda k: -SCORES[k]["macro_F1"])
SETS = {f"top{n}": order[:n] for n in TOP_N_LIST if n <= len(order) or n == 99}
SETS = {("all" if n.endswith("99") else n): v for n, v in SETS.items()}
for g, ks in GROUPS.items():
    ks = [k for k in ks if k in FOLD2]
    if len(ks) >= 2:
        SETS[g] = ks
DOC_IDS = sorted(GOLD2)
HALF = {0: DOC_IDS[0::2], 1: DOC_IDS[1::2]}


def member_weights(keys, wmode):
    return [1.0] * len(keys) if wmode == "equal" else [SCORES[k]["macro_F1"] for k in keys]


def run_config(cfg, docs):
    setname, mode, wmode, tau, dot = cfg
    keys = SETS[setname]
    src = FOLD2_NORM if dot else FOLD2
    return vote([src[k] for k in keys], member_weights(keys, wmode), tau, mode, docs)


def score_docs(preds, docs):
    return strict_scores({d: GOLD2[d] for d in docs}, preds, LABELS)["macro_F1"]


CONFIGS = [(s, m, w, t, dt) for s in SETS for m in MODES for w in WEIGHTS for t in TAUS for dt in DOT_RULES]
t0 = time.time()
GRID = []
for cfg in CONFIGS:
    p = run_config(cfg, DOC_IDS)
    GRID.append((strict_scores(GOLD2, p, LABELS)["macro_F1"], cfg, score_docs(p, HALF[0]), score_docs(p, HALF[1])))
GRID.sort(key=lambda r: -r[0])
print(f"{len(CONFIGS)} configurations evaluated in {time.time()-t0:.0f}s. Best 15 on fold2 (set, mode, weights, tau, dot-rule):")
for f, cfg, h0, h1 in GRID[:15]:
    print(f"  {f:.4f}  (half A {h0:.4f}, half B {h1:.4f})  {cfg}")
print("\nbest per member set:")
for s in SETS:
    r = next(r for r in GRID if r[1][0] == s)
    print(f"  {s:14s} {r[0]:.4f}  {r[1][1:]}  ({len(SETS[s])} members)")
print("\nbest per mode / weighting / dot rule:")
for idx, name in ((1, "mode"), (2, "weights"), (4, "dot rule")):
    for v in sorted({r[1][idx] for r in GRID}, key=str):
        r = next(r for r in GRID if r[1][idx] == v)
        print(f"  {name} = {str(v):9s} best {r[0]:.4f}  {r[1]}")

# honest estimate: choose the configuration on one half of the fold2 documents, apply it to the other half
honest = {}
for a in (0, 1):
    other = 1 - a
    best_cfg = max(CONFIGS, key=lambda c: score_docs(run_config(c, HALF[other]), HALF[other]))
    honest.update(run_config(best_cfg, HALF[a]))
    print(f"  half {a}: configuration chosen on half {other}: {best_cfg}")
HONEST_F1 = strict_scores(GOLD2, honest, LABELS)["macro_F1"]
print(f"\nHONEST estimate of the tuned ensemble (configuration picked on the other half of the documents): {HONEST_F1:.4f}"
      f"  vs best single model {SCORES[BEST_SINGLE]['macro_F1']:.4f}  vs tuned-on-all-of-fold2 {GRID[0][0]:.4f}")


## 8. Report of the chosen ensemble

In [ ]:
# ---- the chosen ensemble: report -------------------------------------------------------------------------------------
BEST_CFG = GRID[0][1]
bp = run_config(BEST_CFG, DOC_IDS)
S = strict_scores(GOLD2, bp, LABELS)
S1 = SCORES[BEST_SINGLE]
print("chosen configuration:", BEST_CFG, "| members:", SETS[BEST_CFG[0]])
print(f"fold2 strict macro-F1 {S['macro_F1']:.4f} (P {S['macro_P']:.4f}, R {S['macro_R']:.4f})  vs best single {BEST_SINGLE} {S1['macro_F1']:.4f} (P {S1['macro_P']:.4f}, R {S1['macro_R']:.4f})")
print(f"{'label':10s} {'ens F1':>7s} {'single F1':>9s} {'ens P':>6s} {'ens R':>6s}")
for l in LABELS:
    print(f"{l:10s} {S['per_label'][l]['F1']:7.3f} {S1['per_label'][l]['F1']:9.3f} {S['per_label'][l]['P']:6.3f} {S['per_label'][l]['R']:6.3f}")
print("errors, ensemble:", error_breakdown(GOLD2, bp))
print("errors, best single:", error_breakdown(GOLD2, FOLD2[BEST_SINGLE]))
print("predicted spans: ensemble", sum(len(v) for v in bp.values()), "| best single", sum(len(v) for v in FOLD2[BEST_SINGLE].values()),
      "| gold", sum(len(v) for v in GOLD2.values()))
# distinct configurations worth a second submission: the best one that uses a different member set
SECOND = next((r for r in GRID if r[1][0] != BEST_CFG[0]), None)
print("best configuration with a different member set:", SECOND[:2] if SECOND else None)


## 9. Test predictions from the zips, submission zips

In [ ]:
# ---- test predictions: ensemble the members' submission zips ---------------------------------------------------------
import glob, re
ZIPS = {}
for z in (glob.glob("/kaggle/input/**/*.zip", recursive=True) + glob.glob("/kaggle/working/*.zip")
          + glob.glob("/kaggle/input/**/*.json", recursive=True)):
    name = os.path.basename(z).rsplit(".", 1)[0]
    if name == "prediction":                                   # Kaggle may unzip an uploaded zip into a folder named after it
        name = os.path.basename(os.path.dirname(z))
    ZIPS[name[len("submission_"):] if name.startswith("submission_") else name] = z
print(f"{len(ZIPS)} zip files found under /kaggle/input:", sorted(ZIPS))
TEST_PRED = {}
def find_zip(zname):
    """Exact name first; else a zip whose name is a truncated prefix of it (Codabench shortens long file names to ~35 chars,
    e.g. 'p2_15b_kd_moe13_to_lstm_plus_unlabe'); a trailing ' (1)' / '_1' from repeated downloads is ignored."""
    if zname in ZIPS:
        return ZIPS[zname]
    cands = []
    for k, path in ZIPS.items():
        base = re.sub(r"(\s*\(\d+\)|_\d+)$", "", k)
        if base == zname or (len(base) >= 30 and zname.startswith(base)):
            cands.append((len(base), path))
    return max(cands)[1] if cands else None


for key, fname, zname in MEMBERS:
    zpath = find_zip(zname)
    if zpath is not None:
        print(f"  {key}: using {zpath.replace('/kaggle/input/', '')}")
        if zpath.endswith(".json"):                       # a bare prediction.json renamed after its model
            p = json.load(open(zpath))
        else:
            with zipfile.ZipFile(zpath) as zf:
                p = json.loads(zf.read("prediction.json"))
        assert set(p) == set(TEST_TEXTS), f"{zname}: document ids differ from the test set"
        TEST_PRED[key] = {d: [tuple(x[:3]) for x in p[d]] for d in p}
missing = [k for k, _, z in MEMBERS if k in FOLD2 and k not in TEST_PRED]
print("members with a test prediction:", len(TEST_PRED), "| missing zips for:", missing)

DOT_ALL = learn_dot_rule({**TEXTS["fold1"], **TEXTS["fold2"]}, {**GOLDS["fold1"], **GOLDS["fold2"]})


def validate_prediction(pred, texts, labels):
    assert set(pred) == set(texts), f"id mismatch: missing {len(set(texts)-set(pred))}, extra {len(set(pred)-set(texts))}"
    for did, ents in pred.items():
        seen = set()
        for x in ents:
            s, e, l = x
            assert type(s) is int and type(e) is int and 0 <= s < e <= len(texts[did]) and l in labels, (did, x)
            assert (s, e, l) not in seen, ("duplicate", did, x); seen.add((s, e, l))


def make_zip(cfg, tag):
    setname, mode, wmode, tau, dot = cfg
    keys = [k for k in SETS[setname] if k in TEST_PRED]
    if len(keys) < 2:
        print(f"  {tag}: only {len(keys)} member(s) with a test prediction -> not written"); return None
    src = {k: (apply_dot_rule(TEST_TEXTS, TEST_PRED[k], DOT_ALL) if dot else TEST_PRED[k]) for k in keys}
    # the vote threshold is a SHARE of the total weight, so it is unchanged when a member has no zip
    pred = vote([src[k] for k in keys], member_weights(keys, wmode), tau, mode, sorted(TEST_TEXTS))
    pred = {d: [[int(s), int(e), l] for (s, e, l) in v] for d, v in pred.items()}
    validate_prediction(pred, TEST_TEXTS, LABELS)
    with open(f"{WORK}/prediction.json", "w", encoding="utf-8") as f:
        json.dump(pred, f)
    zp = f"{WORK}/submission_{RUN_NAME}_{tag}.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{WORK}/prediction.json", arcname="prediction.json")
    with zipfile.ZipFile(zp) as z:
        assert z.namelist() == ["prediction.json"]
    n = sum(len(v) for v in pred.values())
    print(f"  wrote {zp}: {n} entities, members {keys}, config {cfg[1:]}")
    return dict(zip=os.path.basename(zp), config=list(cfg), members=keys, entities=n)


WRITTEN = [w for w in (make_zip(BEST_CFG, "A"), make_zip(SECOND[1], "B") if SECOND else None) if w]


## 10. Save and print the result block

In [ ]:
RESULT = dict(run=RUN_NAME, members_fold2={k: round(v["macro_F1"], 4) for k, v in SCORES.items()}, best_single=BEST_SINGLE,
              best_config=list(BEST_CFG), best_fold2_F1=round(S["macro_F1"], 4), honest_F1=round(HONEST_F1, 4),
              per_label_F1={l: round(S["per_label"][l]["F1"], 4) for l in LABELS}, errors=error_breakdown(GOLD2, bp),
              top_grid=[(round(r[0], 4), list(r[1])) for r in GRID[:30]], written=WRITTEN,
              notebook_minutes=round((time.time() - NB_START) / 60, 1))
rp = f"{WORK}/{RUN_NAME}_results.json"
json.dump(RESULT, open(rp, "w"), indent=1, default=str)
upload_model_file(rp)
print("=" * 100)
print(f"RESULT {RUN_NAME}: fold2 {RESULT['best_fold2_F1']} (honest {RESULT['honest_F1']}) vs best single {BEST_SINGLE} {round(SCORES[BEST_SINGLE]['macro_F1'], 4)}; zips: {[w['zip'] for w in WRITTEN]}")
print("=" * 100)
print("SUMMARY_JSON_START"); print(json.dumps(RESULT, default=str)); print("SUMMARY_JSON_END")
